<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector


*Code that actually builds it engineered features, categorical handling, fills.*

Feature vector built from March 2026 (`fact_content_daily_performance`,
partition `month=2026-03`), grain rolled up from page-day to page-month
(one row per `content_hash_id` + `client_hash_id`).

Engineered features: totals/averages aggregated across the month, plus one
ratio (`ctr`) computed from two summed columns rather than averaged daily
ratios (averaging daily CTRs would weight low-traffic days equally with
high-traffic days — summing first avoids that distortion).

Missing-value handling: GSC columns (`impressions`, `clicks`, `avg_position`)
are never null for a page that has any March rows, so no fill needed there.
GA4 columns are null/unavailable for the ~95.8% of rows where
`ga4_data_available` is false — handled explicitly with `fillna(0)` for
counts and a separate boolean flag, rather than silently treating "no GA4
data" the same as "zero engagement" (see Section 2 for why this matters).

In [11]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr,
        AVG(gsc_avg_position) AS avg_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN scroll_events ELSE 0 END) AS scroll_events,
        COALESCE(BOOL_OR(ga4_data_available IS TRUE), FALSE) AS has_any_ga4_data,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS n_days_ga4_flag_null,
        COUNT(*) AS days_with_data
    FROM {DAILY}
    GROUP BY content_hash_id, client_hash_id
""").df()

features["ctr"] = features["ctr"].fillna(0)
features["avg_position"] = features["avg_position"].fillna(0)

print(f"Rows: {len(features)} | Pages: {features['content_hash_id'].nunique()} | Clients: {features['client_hash_id'].nunique()}")
print()
print("has_any_ga4_data (now with explicit IS TRUE / COALESCE, no NULL leakage):")
print(features["has_any_ga4_data"].value_counts(dropna=False))
print()
print(f"Pages where ga4_data_available was NULL on every March day: {(features['n_days_ga4_flag_null'] == features['days_with_data']).sum()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331437 | Pages: 331437 | Clients: 55

has_any_ga4_data (now with explicit IS TRUE / COALESCE, no NULL leakage):
has_any_ga4_data
False    240948
True      90489
Name: count, dtype: int64

Pages where ga4_data_available was NULL on every March day: 70700


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and
whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available-when? |
|---|---|---|---|
| `impressions` | Total GSC impressions, March | Never missing for a page with any March rows | Fully knowable at end of March — trailing, logged data |
| `clicks` | Total GSC clicks, March | Never missing | Trailing, knowable at end of March |
| `ctr` | `clicks / impressions` (%), summed-then-divided | 0 if impressions is 0 | Derived from the two above — same availability |
| `avg_position` | Mean GSC position across March days | 0 if no rows (shouldn't occur given the join) | Trailing GSC data |
| `ga4_sessions` | Sessions, GA4-sourced | 0 whenever `ga4_data_available` is not explicitly TRUE (covers both FALSE and NULL) | Trailing, only for the subset with confirmed GA4 coverage |
| `ga4_engaged_sessions` | Engaged sessions, GA4-sourced | Same treatment | Same caveat |
| `scroll_events` | Scroll interaction count, GA4-sourced | Same treatment | Same caveat |
| `has_any_ga4_data` | Boolean: at least one March day with `ga4_data_available IS TRUE`? Computed with `COALESCE(..., FALSE)` so NULL never silently drops out of the True/False split | Not applicable | Available at end of March |
| `n_days_ga4_flag_null` | Count of March days where the availability flag itself was NULL (unrecorded, not "false") | Not applicable | Available at end of March |

**No categorical features here** — `content_hash_id` and `client_hash_id`
are pseudonymous identifiers used for grouping/joining only, never as
model inputs.

**Data-quality finding (caught while building this section):** an
earlier version of the `has_any_ga4_data` aggregation used `MAX()` without
explicit NULL handling. DuckDB's `MAX()` over an all-NULL group returns
NULL rather than FALSE — so pages where `ga4_data_available` was never
recorded (neither TRUE nor FALSE) were silently falling into neither the
True nor False bucket in a naive `value_counts()`. Fixed with
`COALESCE(BOOL_OR(ga4_data_available IS TRUE), FALSE)`.

**The real finding this surfaced:** 70,700 pages (21.3%) have
`ga4_data_available` as NULL on every single day in March — not "false,"
genuinely unrecorded. Combined with the 240,948 pages (72.7%) explicitly
marked FALSE, only 90,489 pages (27.3%) have any confirmed GA4
measurement at all. This is a stronger, more precise version of Week 3's
original GA4-sparsity finding — GA4 isn't just sparse, a meaningful chunk
of "no GA4"

In [12]:
# Confirm the three-way split adds up cleanly and quantify it precisely.

n_confirmed_true = int(features["has_any_ga4_data"].sum())
n_all_null = int((features["n_days_ga4_flag_null"] == features["days_with_data"]).sum())
n_confirmed_false = len(features) - n_confirmed_true - n_all_null

print(f"Confirmed GA4 available (>=1 day TRUE):        {n_confirmed_true} ({100*n_confirmed_true/len(features):.1f}%)")
print(f"Confirmed GA4 not available (all days FALSE):  {n_confirmed_false} ({100*n_confirmed_false/len(features):.1f}%)")
print(f"GA4 status entirely unrecorded (all days NULL): {n_all_null} ({100*n_all_null/len(features):.1f}%)")
print(f"Sum check: {n_confirmed_true + n_confirmed_false + n_all_null} (should equal {len(features)})")

Confirmed GA4 available (>=1 day TRUE):        90489 (27.3%)
Confirmed GA4 not available (all days FALSE):  170248 (51.4%)
GA4 status entirely unrecorded (all days NULL): 70700 (21.3%)
Sum check: 331437 (should equal 331437)


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product
flags. Show the test.*

**Timeline check:** every feature in Section 1 is aggregated entirely from
March 2026 daily rows. None require April or later data — confirmed in
Section 2's "available-when" column.

**Label-derived check:** this notebook doesn't define a label itself (no
target column built here), so there's no label to check features against
directly. The relevant check is forward-looking: if a label were later
built as "did this page decline in April vs. March" (the same pattern used
in the capstone), then `impressions`/`clicks` here (March totals) would be
one legitimate side of that comparison — expected, not leakage — but a
feature built from April data would be actual leakage. This notebook's
features are month-scoped and safe for that use.

**Product-flag / decision-derived check:** none of the nine features are
outputs of an existing scoring system, prior model, or manual review flag.
All are raw GSC/GA4 metrics or pipeline-quality indicators
(`has_any_ga4_data`, `n_days_ga4_flag_null`) — the latter two describe data
completeness, not content performance, and would need to be excluded from
any model (see Section 4) even though they're not leakage in the strict
sense.

**Quantitative test result:** an initial raw exact-match check between March
`impressions` and April `impressions_april` showed a surprisingly high
42.2% match rate — investigated further before accepting it. Excluding the
154,699 pages with zero March impressions (a large share of the dataset),
the match rate among pages with real traffic drops to 1.8%, consistent with
ordinary chance rather than a leakage bug. Of the zero-impression pages,
88.4% are also zero in April — a plausible, non-leaky pattern (a page with
no March traffic likely wasn't indexed or ranking at that time, and that
status often persists into the next month). **Conclusion: no evidence that
March's feature aggregation accidentally includes April data.**

In [13]:
# Deliberate-leak-style check: pull April data and confirm none of our March
# features accidentally match/duplicate an April aggregate.

APRIL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')"

april_check = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM {APRIL}
    GROUP BY content_hash_id
""").df()

merged_check = features.merge(april_check, on="content_hash_id", how="inner")

identical_count = (merged_check["impressions"] == merged_check["impressions_april"]).sum()
corr = merged_check["impressions"].corr(merged_check["impressions_april"])

print(f"Pages present in both March and April: {len(merged_check)}")
print(f"March 'impressions' exactly equal to April 'impressions_april': {identical_count} rows ({100*identical_count/len(merged_check):.1f}%)")
print(f"Correlation (all pages, zero-inflated): {round(corr, 3)}")
print()

# Rule out the low-traffic zero-match explanation, since a raw match rate this
# high is worth interrogating rather than accepting at face value.
nonzero_check = merged_check[merged_check["impressions"] > 0]
identical_nonzero = (nonzero_check["impressions"] == nonzero_check["impressions_april"]).sum()
zero_check = merged_check[merged_check["impressions"] == 0]
identical_zero = (zero_check["impressions"] == zero_check["impressions_april"]).sum()

print(f"Excluding zero-impression pages — nonzero exact matches: {identical_nonzero} / {len(nonzero_check)} ({100*identical_nonzero/len(nonzero_check):.1f}%)")
print(f"Zero-impression pages where April is ALSO zero: {identical_zero} / {len(zero_check)} ({100*identical_zero/max(len(zero_check),1):.1f}%)")
print()
print("Conclusion: the raw 42.2% match rate was driven almost entirely by")
print("zero-impression pages (154,699 of them, 88.4% of which are also zero")
print("in April — plausible, since pages with no March traffic likely weren't")
print("indexed/ranking then either). Excluding zeros, the real exact-match")
print("rate is 1.8% — consistent with ordinary chance, not a leakage bug.")
print("March's feature query does not appear to be scanning April data.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages present in both March and April: 331436
March 'impressions' exactly equal to April 'impressions_april': 139955 rows (42.2%)
Correlation (all pages, zero-inflated): 0.871

Excluding zero-impression pages — nonzero exact matches: 3148 / 176737 (1.8%)
Zero-impression pages where April is ALSO zero: 136807 / 154699 (88.4%)

Conclusion: the raw 42.2% match rate was driven almost entirely by
zero-impression pages (154,699 of them, 88.4% of which are also zero
in April — plausible, since pages with no March traffic likely weren't
indexed/ranking then either). Excluding zeros, the real exact-match
rate is 1.8% — consistent with ordinary chance, not a leakage bug.
March's feature query does not appear to be scanning April data.


## 4. What I excluded and why

*The list of fields you refused to use with one line of why each.*

| Field / column | Excluded because |
|---|---|
| `content_hash_id`, `client_hash_id` | Pseudonymous identifiers — used only for grouping/joining, never as model inputs (per `flyrank-data` skill's explicit rule). |
| `has_any_ga4_data` | A data-completeness flag describing pipeline coverage, not content performance — including it as a feature risks the model learning "which pages have GA4 tracking" rather than anything about the page itself, and that tracking-coverage pattern isn't guaranteed to persist into future months. |
| `n_days_ga4_flag_null` | Same reasoning as above — a pipeline-quality artifact, not a content signal. Useful for this audit, not for a model. |
| `ga4_sessions`, `ga4_engaged_sessions`, `scroll_events` | Only confirmed-available for 27.3% of pages (Section 2/3 finding); building a feature on a signal this sparse and this unevenly distributed risks the model implicitly learning "is this a GA4-tracked page" as a proxy for something else, rather than a genuine engagement signal. Kept in the feature vector for now as raw columns, but flagged as unsafe to use directly in a model without much more careful handling (e.g. a separate coverage-aware model, or restricting to the GA4-confirmed subset only). |
| Any raw URL, title, query text, or client name | Not present in any table pulled into this notebook — public-safety rule, no exception considered. |
| `report_date` (row-level) | Aggregated away entirely — this notebook works at the page-month grain, not page-day, so no daily date field survives into the feature vector. |
| Any column from `fact_content_query_90d` (query-level table) | Out of scope for this feature vector — a different grain (query, not page-month) that would need its own join/aggregation logic and wasn't needed for this exercise. |

**No product-derived flags or prior model scores exist in this warehouse
release at all** — there was nothing of that category to consider
excluding; noted here only because the self-check explicitly asks about it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.